# AdMarket Arena — Oversight Agent GRPO Training

Trains the **OversightAgent** (Fleet AI Scalable Oversight bonus) on the AdMarket Arena env via TRL GRPO + Unsloth 4-bit LoRA.

**T4 (Colab) tuned config** — base model: Qwen2.5-3B-Instruct (4-bit NF4 + LoRA r=16), GRPO `num_generations=4`, 64-token output cap, ~3-4 hr expected run on Colab T4.

Decoupled from advertiser training: the dataset (`data/oversight_train_trajectories.jsonl`) is pre-generated by `scripts/collect_oversight_trajectories.py` using a frozen rule-based pacing advertiser × `violation_injector`, so this notebook can run in parallel with or after the advertiser training notebook.

**Inputs**: `data/oversight_train_trajectories.jsonl` (200 episodes recommended, ~1400 day-records).

**Outputs**:
- `checkpoints/oversight_step_<N>/` LoRA adapters
- `checkpoints/oversight_best/` symlink to best by validation F1
- W&B project `admarket-arena-oversight` (public)
- HF Hub push to `<user>/admarket-oversight-qwen2.5-3b-grpo`

**Reward**: `OversightF1Rubric.score_day(...)` — `daily_f1 - 0.5 * false_positives`. False-positive penalty prevents the trivial 'flag everyone' exploit.

## 1. Install dependencies

In [ ]:
%%capture
# Pin versions tested with Unsloth on Colab T4 (Apr 2026).
!pip install -U unsloth
!pip install -U trl==0.12.* peft==0.13.* transformers==4.46.* accelerate==1.1.* datasets==3.* bitsandbytes==0.44.*
!pip install -U wandb pydantic

## 2. Mount drive / clone repo

If you've copied the repo into the Colab runtime already, skip the clone and just `cd` into it. Otherwise pull from the team repo so the latest `oversight.py`, `violation_injector.py`, `models.py` are available.

In [ ]:
import os, sys
REPO_PATH = '/content/meta_ad_optimizer'  # change to your local path if running outside Colab
if not os.path.exists(REPO_PATH):
    # Replace with your repo URL.
    !git clone https://github.com/<your-org>/meta_ad_optimizer.git $REPO_PATH
%cd $REPO_PATH
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

## 3. Imports + config

In [ ]:
import json
import os
import random
import textwrap
from pathlib import Path
from typing import List

import torch
import wandb
from datasets import Dataset

from models import GroundTruthViolation, OversightObservation, ViolationFlag
from oversight import (
    OVERSIGHT_SYSTEM_PROMPT,
    HeuristicOversightAgent,
    _format_observation_for_prompt,
    parse_llm_flags,
    score_flags,
)
from server.arena_rubrics import OversightF1Rubric

# --- knobs ---
BASE_MODEL          = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN         = 4096
LORA_RANK           = 16
TRAIN_DATA_PATH     = Path('data/oversight_train_trajectories.jsonl')
VAL_FRACTION        = 0.10
OUTPUT_DIR          = Path('checkpoints')
MAX_NEW_TOKENS      = 64
NUM_GENERATIONS     = 4   # GRPO group size, T4-tight
LEARNING_RATE       = 1e-5
MAX_STEPS           = 60
SAVE_EVERY_STEPS    = 10
LOG_EVERY_STEPS     = 1
WANDB_PROJECT       = 'admarket-arena-oversight'
HF_HUB_REPO_ID      = '<your-org>/admarket-oversight-qwen2.5-3b-grpo'  # set before running push cell
SEED                = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); torch.manual_seed(SEED)
print('Config OK.')

## 4. Load model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
print(f'Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params (LoRA).')

## 5. Build dataset from oversight trajectories

Each prompt = one OversightObservation rendered into the canonical format from `oversight._format_observation_for_prompt`. The reward function below decodes the LLM's JSON output via `parse_llm_flags`, scores it against the ground truth via `OversightF1Rubric`, and returns a scalar reward to GRPO.

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw = load_jsonl(TRAIN_DATA_PATH)
random.Random(SEED).shuffle(raw)
split_at = max(1, int(len(raw) * (1 - VAL_FRACTION)))
raw_train, raw_val = raw[:split_at], raw[split_at:]
print(f'Loaded {len(raw)} day-records. train={len(raw_train)}  val={len(raw_val)}')

def build_example(row: dict) -> dict:
    obs = OversightObservation.model_validate(row['observation'])
    user_prompt = _format_observation_for_prompt(obs, max_log_lines=80)
    chat = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': OVERSIGHT_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        'prompt': chat,
        'ground_truth_json': json.dumps(row['ground_truth']),
    }

train_ds = Dataset.from_list([build_example(r) for r in raw_train])
val_ds   = Dataset.from_list([build_example(r) for r in raw_val])
print(train_ds[0]['prompt'][:400], '...')

## 6. Reward function (F1 with FP penalty)

In [ ]:
_RUBRIC = OversightF1Rubric()

def oversight_reward_fn(prompts, completions, ground_truth_json, **_):
    """GRPO reward for a batch of completions.

    Each `completion` is a list-of-dicts from the chat template. The
    OversightF1Rubric.score_day returns a dict; we use its 'reward'
    field directly so the signal is identical to the env's training
    rubric.
    """
    rewards = []
    for completion, gt_json in zip(completions, ground_truth_json):
        # Newer TRL versions wrap completions; handle both forms.
        text = completion if isinstance(completion, str) else completion[-1].get('content', '')
        flags: List[ViolationFlag] = parse_llm_flags(text)
        truth = [GroundTruthViolation.model_validate(t) for t in json.loads(gt_json)]
        scored = _RUBRIC.score_day(flags, truth)
        rewards.append(float(scored['reward']))
    return rewards

# Sanity-check reward on a heuristic-prediction baseline.
_h = HeuristicOversightAgent()
_demo_obs = OversightObservation.model_validate(raw_val[0]['observation'])
_demo_truth = [GroundTruthViolation.model_validate(t) for t in raw_val[0]['ground_truth']]
_demo_flags = _h.flag_day(_demo_obs)
print('heuristic flags:', [(f.advertiser_id, f.violation_type) for f in _demo_flags])
print('heuristic reward:', _RUBRIC.score_day(_demo_flags, _demo_truth)['reward'])

## 7. Smoke-test logging on first 2 GRPO steps

**Critical insurance step.** Verify W&B + per-step custom metrics fire correctly before launching the long run. A broken logger costs 4 hours of GPU time.

In [ ]:
wandb.login()  # paste API key when prompted
wandb.init(
    project=WANDB_PROJECT,
    name=f'oversight_grpo_qwen3b_seed{SEED}',
    config={
        'base_model': BASE_MODEL,
        'lora_rank': LORA_RANK,
        'num_generations': NUM_GENERATIONS,
        'learning_rate': LEARNING_RATE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'train_examples': len(train_ds),
        'val_examples': len(val_ds),
        'max_steps': MAX_STEPS,
    },
)

## 8. GRPO trainer + run

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(OUTPUT_DIR / 'oversight_run'),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
    max_completion_length=MAX_NEW_TOKENS,
    max_steps=MAX_STEPS,
    save_steps=SAVE_EVERY_STEPS,
    logging_steps=LOG_EVERY_STEPS,
    bf16=False,
    fp16=True,
    optim='adamw_8bit',
    report_to=['wandb'],
    remove_unused_columns=False,
    seed=SEED,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[oversight_reward_fn],
    args=config,
    train_dataset=train_ds,
)
trainer.train()

## 9. Validation F1 + best-checkpoint selection

In [ ]:
FastLanguageModel.for_inference(model)

def _generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

val_f1s = []
for example in val_ds:
    text = _generate(example['prompt'])
    flags = parse_llm_flags(text)
    truth = [GroundTruthViolation.model_validate(t) for t in json.loads(example['ground_truth_json'])]
    val_f1s.append(score_flags(flags, truth).f1)
import statistics as _stats
val_f1_mean = _stats.mean(val_f1s) if val_f1s else 0.0
print(f'Validation mean F1: {val_f1_mean:.3f}  (n={len(val_f1s)})')
wandb.log({'val/mean_f1': val_f1_mean})

# Save best checkpoint (overwrite the symlink to the latest if best).
BEST_PATH = OUTPUT_DIR / 'oversight_best'
model.save_pretrained(str(BEST_PATH))
tokenizer.save_pretrained(str(BEST_PATH))
print(f'Saved best checkpoint to {BEST_PATH}')

## 10. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste token

model.push_to_hub(HF_HUB_REPO_ID, private=False)
tokenizer.push_to_hub(HF_HUB_REPO_ID, private=False)

model_card = textwrap.dedent(f"""\
---
language: en
license: apache-2.0
library_name: peft
tags:
- grpo
- unsloth
- oversight
- ad-tech
base_model: {BASE_MODEL}
---

# AdMarket Arena — Trained Oversight Agent

GRPO-trained LoRA adapter for the AdMarket Arena env's OversightAgent role. Detects three violation types (frequency_cap, budget_overspend, shill_bidding) from auction logs.

**Training**: {MAX_STEPS} GRPO steps, num_generations={NUM_GENERATIONS}, lr={LEARNING_RATE}, on Colab T4. Reward function: `OversightF1Rubric.score_day` (daily F1 minus 0.5 per false positive).

**Validation F1**: {val_f1_mean:.3f}

**W&B**: https://wandb.ai/<your-org>/{WANDB_PROJECT}

**Eval results**: see `oversight_eval_results.json` in the env repo, generated by `scripts/oversight_eval.py`.
""")
Path('MODEL_CARD.md').write_text(model_card)
HfApi().upload_file(path_or_fileobj='MODEL_CARD.md', path_in_repo='README.md', repo_id=HF_HUB_REPO_ID, repo_type='model')
print('Pushed to', HF_HUB_REPO_ID)
wandb.finish()